# Real-Data End-to-End Notebook Workflow
Bu notebook, frameworku tek varlık (**BTC/USDT**) ve çoklu portföy akışı için uçtan uca çalıştırır: ortam kurulumu, gerçek kaynaklardan discovery+ingestion, feature/backtest/optimizasyon, signal->intent->risk->portfolio zinciri ve sonuç metrikleri.

## 1) Ortam kurulumu ve config/custom parametre seçimi
- İsteğe bağlı paketler: `ccxt`, `matplotlib`
- Parametreler: sembol, interval, grid-search aralığı, risk limitleri

In [ ]:
from __future__ import annotations

import csv
import io
import json
import math
import statistics
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG = {
    'exchange_id': 'binance',
    'single_symbol': 'BTC/USDT',
    'symbol_limit': 8,
    'interval': '1h',
    'candle_limit': 300,
    'short_windows': [5, 10, 20],
    'long_windows': [30, 50, 80],
    'train_ratio': 0.7,
    'initial_cash': 10000.0,
    'max_notional_per_trade': 2500.0,
}
CONFIG


## 2) Data/asset discovery ve gerçek kaynaklardan ingestion
- ccxt mevcutsa doğrudan exchange symbol discovery yapılır
- Kline ingestion için Binance public endpoint kullanılır
- Haber ve makro örnek ingest için public RSS/FRED çekimi yapılır

In [ ]:
def _http_get(url: str, timeout: int = 20) -> str:
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return response.read().decode('utf-8')


def discover_assets(exchange_id: str, fallback_symbols: list[str]) -> list[str]:
    try:
        import ccxt  # type: ignore

        exchange_cls = getattr(ccxt, exchange_id)
        exchange = exchange_cls({'enableRateLimit': True})
        markets = exchange.load_markets()
        symbols = sorted([symbol for symbol in markets if '/' in symbol])
        return symbols
    except Exception as exc:  # pragma: no cover - notebook runtime guard
        print(f'ccxt discovery kullanılamadı, fallback liste ile devam: {exc}')
        return fallback_symbols


def fetch_binance_klines(symbol: str, interval: str, limit: int) -> list[dict[str, Any]]:
    normalized = symbol.replace('/', '')
    query = urllib.parse.urlencode({'symbol': normalized, 'interval': interval, 'limit': limit})
    url = f'https://api.binance.com/api/v3/klines?{query}'
    raw = json.loads(_http_get(url))
    rows: list[dict[str, Any]] = []
    for item in raw:
        rows.append({
            'symbol': symbol,
            'open_time': int(item[0]),
            'open': float(item[1]),
            'high': float(item[2]),
            'low': float(item[3]),
            'close': float(item[4]),
            'volume': float(item[5]),
        })
    return rows


def ingest_macro_and_news() -> dict[str, Any]:
    # Macro: FRED CPI public CSV
    macro_csv = _http_get('https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL')
    macro_rows = list(csv.DictReader(io.StringIO(macro_csv)))

    # News: CoinDesk RSS public feed
    rss_xml = _http_get('https://www.coindesk.com/arc/outboundfeeds/rss/')
    xml_root = ET.fromstring(rss_xml)
    titles: list[str] = []
    for item in xml_root.findall('.//item')[:5]:
        title = item.findtext('title')
        if title:
            titles.append(title.strip())

    return {
        'macro_latest': macro_rows[-1] if macro_rows else {},
        'news_headlines': titles,
    }


symbol_fallback = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'SOL/USDT']
discovered_symbols = discover_assets(CONFIG['exchange_id'], fallback_symbols=symbol_fallback)
portfolio_symbols = [s for s in discovered_symbols if s.endswith('/USDT')][: CONFIG['symbol_limit']]

single_asset_rows = fetch_binance_klines(CONFIG['single_symbol'], CONFIG['interval'], CONFIG['candle_limit'])
macro_news_snapshot = ingest_macro_and_news()

print('discovered_count=', len(discovered_symbols))
print('portfolio_symbols=', portfolio_symbols)
print('single_asset_rows=', len(single_asset_rows))
print('macro_latest=', macro_news_snapshot['macro_latest'])
print('news_headlines_count=', len(macro_news_snapshot['news_headlines']))


## 3) Feature engineering, dataset inceleme ve görselleştirme
Aşağıdaki hücre close fiyatından rolling ortalamalar ve getiriler üretir; `matplotlib` varsa PnL/price eğrisi çizilir, yoksa metinsel özet gösterilir.

In [ ]:
def rolling_mean(values: list[float], window: int) -> list[float | None]:
    out: list[float | None] = []
    for idx in range(len(values)):
        if idx + 1 < window:
            out.append(None)
            continue
        segment = values[idx + 1 - window : idx + 1]
        out.append(sum(segment) / window)
    return out


def with_features(rows: list[dict[str, Any]], short: int, long: int) -> list[dict[str, Any]]:
    closes = [r['close'] for r in rows]
    sma_short = rolling_mean(closes, short)
    sma_long = rolling_mean(closes, long)
    enriched: list[dict[str, Any]] = []
    for i, row in enumerate(rows):
        prev_close = closes[i - 1] if i > 0 else closes[i]
        ret = 0.0 if prev_close == 0 else (closes[i] - prev_close) / prev_close
        signal = 0
        if sma_short[i] is not None and sma_long[i] is not None:
            signal = 1 if sma_short[i] > sma_long[i] else -1
        enriched.append({
            **row,
            'return': ret,
            'sma_short': sma_short[i],
            'sma_long': sma_long[i],
            'signal': signal,
        })
    return enriched


@dataclass
class BacktestResult:
    symbol: str
    short_window: int
    long_window: int
    train_pnl: float
    oos_pnl: float
    trade_count: int
    pnl_curve: list[float]
    trade_log: list[dict[str, Any]]


def run_backtest(rows: list[dict[str, Any]], short: int, long: int, train_ratio: float) -> BacktestResult:
    data = with_features(rows, short=short, long=long)
    split = max(2, int(len(data) * train_ratio))
    train, oos = data[:split], data[split:]

    def _simulate(batch: list[dict[str, Any]]) -> tuple[float, int, list[float], list[dict[str, Any]]]:
        equity = CONFIG['initial_cash']
        curve = [equity]
        trades = 0
        log: list[dict[str, Any]] = []
        for row in batch:
            exposure = row['signal']
            pnl_change = equity * exposure * row['return']
            equity += pnl_change
            curve.append(equity)
            if exposure != 0:
                trades += 1
                log.append({'symbol': row['symbol'], 'open_time': row['open_time'], 'signal': exposure, 'equity': equity})
        return equity - CONFIG['initial_cash'], trades, curve, log

    train_pnl, _, _, _ = _simulate(train)
    oos_pnl, trades, oos_curve, trade_log = _simulate(oos)
    return BacktestResult(
        symbol=rows[0]['symbol'],
        short_window=short,
        long_window=long,
        train_pnl=train_pnl,
        oos_pnl=oos_pnl,
        trade_count=trades,
        pnl_curve=oos_curve,
        trade_log=trade_log,
    )


def optimize_windows(rows: list[dict[str, Any]]) -> BacktestResult:
    candidates: list[BacktestResult] = []
    for short in CONFIG['short_windows']:
        for long in CONFIG['long_windows']:
            if short >= long:
                continue
            candidates.append(run_backtest(rows, short=short, long=long, train_ratio=CONFIG['train_ratio']))
    if not candidates:
        raise RuntimeError('No optimization candidates were produced.')
    return sorted(candidates, key=lambda c: (c.oos_pnl, c.train_pnl), reverse=True)[0]


best_single = optimize_windows(single_asset_rows)
print(best_single)


## 4) Strateji instantiate, rolling/OOS backtest ve tek varlık için signal -> intent -> risk -> portfolio
Bu adımda frameworkün `TradeFlow` orkestrasyonunu kullanıyoruz.

In [ ]:
from src.algotradeplan.orchestration.trade_flow import TradeFlow


class RollingSignalStrategy:
    plugin_id = 'rolling_signal_strategy'

    def __init__(self, signal: int) -> None:
        self.signal = signal

    def generate_signal(self, context: dict[str, Any]) -> dict[str, Any]:
        action = 'hold'
        if self.signal > 0:
            action = 'buy'
        elif self.signal < 0:
            action = 'sell'
        return {'action': action, 'context': context}


class MaxNotionalRisk:
    plugin_id = 'max_notional_risk'

    def evaluate(self, order_intent: dict[str, Any]) -> dict[str, Any]:
        price = float(order_intent['context']['close'])
        qty = float(order_intent['context'].get('qty', 1.0))
        notional = price * qty
        approved = notional <= CONFIG['max_notional_per_trade']
        return {'approved': approved, 'notional': notional, 'reason': 'ok' if approved else 'max_notional'}


class RecordingExecution:
    plugin_id = 'recording_execution'

    def __init__(self) -> None:
        self.orders: list[dict[str, Any]] = []

    def send_order(self, order: dict[str, Any]) -> dict[str, Any]:
        self.orders.append(order)
        return {'status': 'accepted', 'order': order}


class PortfolioManager:
    def __init__(self, initial_cash: float) -> None:
        self.cash = initial_cash
        self.positions: dict[str, float] = {}
        self.history: list[dict[str, Any]] = []

    def apply_execution(self, execution_result: dict[str, Any] | None) -> None:
        if not execution_result:
            return
        order = execution_result['order']
        symbol = str(order['symbol'])
        price = float(order['context']['close'])
        qty = float(order['context'].get('qty', 1.0))
        action = str(order['action'])
        direction = 1.0 if action == 'buy' else -1.0
        self.positions[symbol] = self.positions.get(symbol, 0.0) + direction * qty
        self.cash -= direction * qty * price
        self.history.append({'symbol': symbol, 'action': action, 'qty': qty, 'price': price, 'cash': self.cash})

    def summary(self) -> dict[str, Any]:
        return {'cash': self.cash, 'positions': dict(self.positions), 'fills': len(self.history)}


latest = with_features(single_asset_rows, best_single.short_window, best_single.long_window)[-1]
strategy = RollingSignalStrategy(signal=int(latest['signal']))
risk = MaxNotionalRisk()
execution = RecordingExecution()
flow = TradeFlow(strategy=strategy, risk=risk, execution=execution)

flow_result = flow.run({'symbol': CONFIG['single_symbol'].replace('/', ''), 'close': latest['close'], 'qty': 1.0})
portfolio = PortfolioManager(initial_cash=CONFIG['initial_cash'])
portfolio.apply_execution(flow_result.execution)

print('flow_result=', flow_result)
print('portfolio_summary=', portfolio.summary())
print('trade_log_sample=', best_single.trade_log[:3])

try:
    import matplotlib.pyplot as plt  # type: ignore

    plt.figure(figsize=(8, 3))
    plt.plot(best_single.pnl_curve)
    plt.title('Single-asset OOS PnL curve')
    plt.xlabel('Step')
    plt.ylabel('Equity')
    plt.show()
except Exception as exc:  # pragma: no cover - notebook runtime guard
    print('matplotlib mevcut değil, eğri değerleri metin olarak gösteriliyor:', best_single.pnl_curve[:10], '...', exc)


## 5) Tüm portföy için çalıştırma ve metrik özeti
Bu bölümde discovery ile gelen çoklu varlıkta aynı akış koşulur ve portföy seviyesinde PnL/trade sayısı raporlanır.

In [ ]:
portfolio_results: list[BacktestResult] = []
for symbol in portfolio_symbols:
    try:
        rows = fetch_binance_klines(symbol, CONFIG['interval'], CONFIG['candle_limit'])
        portfolio_results.append(optimize_windows(rows))
    except Exception as exc:
        print(f'[{symbol}] ingest/backtest atlandı: {exc}')

if not portfolio_results:
    raise RuntimeError('Portföy için sonuç üretilemedi. Ağ erişimi ve sembolleri kontrol edin.')

total_oos_pnl = sum(r.oos_pnl for r in portfolio_results)
total_trades = sum(r.trade_count for r in portfolio_results)
best_symbol = sorted(portfolio_results, key=lambda x: x.oos_pnl, reverse=True)[0].symbol

summary = {
    'asset_count': len(portfolio_results),
    'total_oos_pnl': round(total_oos_pnl, 2),
    'total_trades': total_trades,
    'best_symbol_by_oos_pnl': best_symbol,
}
summary


## 6) Sonuçlar ve sonraki adımlar
- Tek varlık ve portföy seviyesinde discovery->ingestion->feature->optimize/backtest->intent/risk/portfolio zinciri tamamlandı.
- Bu notebook onboarding akışına ve CI notebook smoke kontrolüne bağlıdır (`make smoke`).
- Üretimde: API key yönetimi, gecikme/slippage modelinin geliştirilmesi ve execution connector hardening adımları eklenmelidir.